In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [3]:
load_dotenv(override=True)
api_key = os.getenv('GROQ_API_KEY')

if api_key and api_key.startswith('gsk_') and len(api_key)>10:
  print("API key looks good so far")
else:
  print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

MODEL = 'openai/gpt-oss-20b'
openai = OpenAI(
  api_key=os.getenv("GROQ_API_KEY"),
  base_url="https://api.groq.com/openai/v1"
)

API key looks good so far


In [5]:
links = fetch_website_links("https://anthropic.com")
links

['#main',
 '#footer',
 'https://www.anthropic.com/',
 'https://www.anthropic.com/research',
 'https://www.anthropic.com/research/team/alignment',
 'https://www.anthropic.com/research/team/economics',
 'https://www.anthropic.com/engineering',
 'https://www.anthropic.com/research/team/frontier-red-team',
 'https://www.anthropic.com/research/team/interpretability',
 'https://www.anthropic.com/science',
 'https://www.anthropic.com/research/team/societal-impacts',
 'https://www.anthropic.com/policy',
 'https://www.anthropic.com/constitution',
 'https://www.anthropic.com/claude-corps',
 'https://www.anthropic.com/policy-on-the-ai-exponential',
 'https://www.anthropic.com/transparency',
 'https://www.anthropic.com/responsible-scaling-policy',
 'https://www.anthropic.com/beneficial-deployments',
 'http://trust.anthropic.com/',
 'https://academy.claude.com',
 'https://claude.com/resources/tutorials',
 'https://claude.com/resources/use-cases',
 'https://platform.claude.com/docs',
 'https://www.a

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://anthropic.com"))


Here is the list of links on the website https://anthropic.com -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#main
#footer
https://www.anthropic.com/
https://www.anthropic.com/research
https://www.anthropic.com/research/team/alignment
https://www.anthropic.com/research/team/economics
https://www.anthropic.com/engineering
https://www.anthropic.com/research/team/frontier-red-team
https://www.anthropic.com/research/team/interpretability
https://www.anthropic.com/science
https://www.anthropic.com/research/team/societal-impacts
https://www.anthropic.com/policy
https://www.anthropic.com/constitution
https://www.anthropic.com/claude-corps
https://www.anthropic.com/policy-on-the-ai-exponential
https://www.anthropic.com/transparency
https://www.anthropic.com/responsible-scaling-policy
https://www.anthropic.com/b

In [11]:
def get_relevant_links(url):
  response = openai.chat.completions.create(
    model=MODEL,
    messages=[
      {"role": "system", "content": link_system_prompt},
      {"role": "user", "content": get_links_user_prompt(url)}
    ],
    response_format={"type": "json_object"}
  )
  result = response.choices[0].message.content
  links = json.loads(result)
  print(f"Found {len(links['links'])} relevant links for {url}:")
  return links

In [13]:
get_relevant_links("https://anthropic.com")

Found 6 relevant links for https://anthropic.com:


{'links': [{'type': 'home page', 'url': 'https://www.anthropic.com/'},
  {'type': 'about page', 'url': 'https://www.anthropic.com/company'},
  {'type': 'leadership page',
   'url': 'https://www.anthropic.com/company/leadership'},
  {'type': 'careers page', 'url': 'https://www.anthropic.com/careers'},
  {'type': 'product overview page',
   'url': 'https://claude.com/product/overview'},
  {'type': 'contact sales page', 'url': 'https://claude.com/contact-sales'}]}

Togather

In [14]:
def get_page_and_all_relevant_links(url):
  page_content = fetch_website_contents(url)
  relevant_links = get_relevant_links(url)
  result = f"## Landing page:\n\n{page_content}\n\n## Relevant links:\n"
  for link in relevant_links['links']:
    result += f"\n\n ### Link: {link['type']}\n"
    result += fetch_website_contents(link['url'])
  return result

In [16]:
display(Markdown(get_page_and_all_relevant_links("https://anthropic.com")))

Found 6 relevant links for https://anthropic.com:


## Landing page:

Home \ Anthropic

Skip to main content
Skip to footer
Research
Overview
Alignment
Economics
Engineering
Frontier Red Team
Interpretability
Science
Societal Impacts
Policy
Commitments
Initiatives
Claude's Constitution
Claude Corps
Policy on the AI Exponential
Transparency
Responsible Scaling Policy
Beneficial Deployments
Trust center
Security and compliance
Learn
Learn
Claude Academy
Tutorials
Use cases
Developer docs
Company
About
Leadership
Careers
Events
News
Try Claude
Try Claude
Try Claude
Learn more about Claude
About Claude
Overview
Pricing
Contact sales
Models
Mythos
Fable
Opus
Sonnet
Haiku
Log in
Claude.ai
Claude Console
EN
This is some text inside of a div block.
Log in to Claude
Log in to Claude
Log in to Claude
Download app
Download app
Download app
Research
Overview
Alignment
Economics
Engineering
Frontier Red Team
Interpretability
Science
Societal Impacts
Policy
Commitments
Initiatives
Claude's Constitution
Claude Corps
Policy on the AI Exponential
Transparency
Responsible Scaling Policy
Beneficial Deployments
Trust center
Security and compliance
Learn
Learn
Claude Academy
Tutorials
Use cases
Developer docs
Company
About
Leadership
Careers
Events
News
Try Claude
Try Claude
Try Claude
Learn more about Claude
About Claude
Overview
Pricing
Contact sales
Models
Mythos
Fable
Opus
Sonnet
Haiku
Log in
Claude.ai
Claude Console
EN
This is some text inside of a div block.
Log in to Claude
Log in to Claude
Log in to Claude
Download app
Download app
Download app
AI
research
and
products
that put safety at the frontier
AI will have a vast impact on the world. Anthropic is a public benefit corporation dedicated to securing its benefits and mitigating its risks.
:Claude:
Fable 5.1
and Mythos 5.1
The world’s most advanced models for coding and knowledge work.
Read more
Read more
Read more
Latest releases
Introducing Fable 5.1 and Mythos 5.1
Our most advanced models for coding and knowledge work. Their research capabilities also offer an early glimpse of how AI models w

## Relevant links:


 ### Link: home page
Home \ Anthropic

Skip to main content
Skip to footer
Research
Overview
Alignment
Economics
Engineering
Frontier Red Team
Interpretability
Science
Societal Impacts
Policy
Commitments
Initiatives
Claude's Constitution
Claude Corps
Policy on the AI Exponential
Transparency
Responsible Scaling Policy
Beneficial Deployments
Trust center
Security and compliance
Learn
Learn
Claude Academy
Tutorials
Use cases
Developer docs
Company
About
Leadership
Careers
Events
News
Try Claude
Try Claude
Try Claude
Learn more about Claude
About Claude
Overview
Pricing
Contact sales
Models
Mythos
Fable
Opus
Sonnet
Haiku
Log in
Claude.ai
Claude Console
EN
This is some text inside of a div block.
Log in to Claude
Log in to Claude
Log in to Claude
Download app
Download app
Download app
Research
Overview
Alignment
Economics
Engineering
Frontier Red Team
Interpretability
Science
Societal Impacts
Policy
Commitments
Initiatives
Claude's Constitution
Claude Corps
Policy on the AI Exponential
Transparency
Responsible Scaling Policy
Beneficial Deployments
Trust center
Security and compliance
Learn
Learn
Claude Academy
Tutorials
Use cases
Developer docs
Company
About
Leadership
Careers
Events
News
Try Claude
Try Claude
Try Claude
Learn more about Claude
About Claude
Overview
Pricing
Contact sales
Models
Mythos
Fable
Opus
Sonnet
Haiku
Log in
Claude.ai
Claude Console
EN
This is some text inside of a div block.
Log in to Claude
Log in to Claude
Log in to Claude
Download app
Download app
Download app
AI
research
and
products
that put safety at the frontier
AI will have a vast impact on the world. Anthropic is a public benefit corporation dedicated to securing its benefits and mitigating its risks.
:Claude:
Fable 5.1
and Mythos 5.1
The world’s most advanced models for coding and knowledge work.
Read more
Read more
Read more
Latest releases
Introducing Fable 5.1 and Mythos 5.1
Our most advanced models for coding and knowledge work. Their research capabilities also offer an early glimpse of how AI models w

 ### Link: about page
Company \ Anthropic

Skip to main content
Skip to footer
Research
Policy
Commitments
Learn
News
Try Claude
Making AI systems you can rely on
Anthropic is an AI safety and research company. We build reliable, interpretable, and steerable AI systems.
Join us
Our Purpose
We believe AI will have a vast impact on the world. Anthropic is dedicated to building systems that people can rely on and generating research about the opportunities and risks of AI.
We Build Safer Systems
We aim to build frontier AI systems that are reliable, interpretable, and steerable. We conduct frontier research, develop and apply a variety of safety techniques, and deploy the resulting systems via a set of partnerships and products.
Safety Is a Science
We treat AI safety as a systematic science, conducting research, applying it to our products, feeding those insights back into our research, and regularly sharing what we learn with the world along the way.
Interdisciplinary
Anthropic is a collaborative team of researchers, engineers, policy experts, business leaders and operators, who bring our experience from many different domains to our work.
AI Companies are One Piece of a Big Puzzle
AI has the potential to fundamentally change how the world works. We view ourselves as just one piece of this evolving puzzle. We collaborate with civil society, government, academia, nonprofits and industry to promote safety industry-wide.
The Team
We’re a team of researchers, engineers, policy experts and operational leaders, with experience spanning a variety of disciplines, all working together to build reliable and understandable AI systems.
Research
We conduct frontier AI research across a variety of modalities, and explore novel and emerging safety research areas from interpretability to RL from human feedback to policy and societal impacts analysis.
Policy
We think about the impacts of our work and strive to communicate what we’re seeing at the frontier to policymakers and civil society in the US and abr

 ### Link: leadership page
Leadership at Anthropic \ Anthropic

Skip to main content
Skip to footer
Research
Policy
Commitments
Learn
News
Try Claude
Leadership at Anthropic
As a public benefit corporation, Anthropic is dedicated to ensuring the world safely makes the transition through transformative AI. Our multidisciplinary leadership team shapes and drives the company's strategy in support of this mission.
Learn about Anthropic
Join the team
CEO and President
Co-Founder and Chief Executive Officer
Dario Amodei
Read blog
Dario Amodei co-founded Anthropic and leads the company’s research direction and strategic vision for developing AI systems that are reliable, interpretable, and steerable.
Read full bio
Co-Founder and President
Daniela Amodei
Daniela Amodei co-founded Anthropic and serves as chair of its Board of Directors. As President, she leads the organization’s work across research, engineering, product, governance, and commercial execution and manages the executive team.
Read full bio
Co-Founders
Co-Founder and Chief Compute Officer
Tom Brown
Tom Brown co-founded Anthropic and leads the technical organization responsible for securing, scaling, and effectively using its compute resources.
Read full bio
Co-Founder and Head of Public Benefit
Jack Clark
Jack Clark co-founded Anthropic and leads The Anthropic Institute, a think tank dedicated to producing research on the societal implications of frontier AI.
Read full bio
Co-Founder and Chief Science Officer
Jared Kaplan
Jared Kaplan co-founded Anthropic and heads its technical research efforts, leading foundational work on AI safety and development.
Read full bio
Co-Founder and Chief Architect
Sam McCandlish
Sam McCandlish co-founded Anthropic and focuses on large-scale model training, leading pretraining, research productivity, and RL infrastructure.
Read full bio
Co-Founder and Interpretability Research Lead
Chris Olah
Chris Olah co-founded Anthropic and leads its interpretability research, which focuses on understanding how AI models

 ### Link: careers page
Careers \ Anthropic

Skip to main content
Skip to footer
Research
Policy
Commitments
Learn
News
Try Claude
Shape how AI meets the world
Anthropic builds Claude—AI designed to be helpful, honest, and harmless. We're researchers, engineers, and builders from a range of disciplines, working to make sure powerful AI goes well for everyone. If you're drawn to hard problems with real stakes, we'd like to meet you.
Explore open roles
Building Anthropic
Our co-founders discuss the origins of Anthropic, the “race to the top” in AI development, and where AI technology will go from here.
Principles that guide how we show up for each other and our mission
Act for the global good
We strive to make decisions that maximize positive outcomes for humanity in the long run. This means we’re willing to be very bold in the actions we take to ensure our technology is a robustly positive force for good. We take seriously the task of safely guiding the world through a technological revolution that has the potential to change the course of human history, and are committed to helping make this transition go well.
Hold light and shade
AI has the potential to pose unprecedented risks to humanity if things go badly. It also has the potential to create unprecedented benefits for humanity if things go well. We need shade to understand and protect against the potential for bad outcomes. We need light to realize the good outcomes.
Be good to our users
At Anthropic, we define “users” broadly. Users are our customers, policy-makers, Ants, and anyone impacted by the technology we build or the actions we take. We cultivate generosity and kindness in all our interactions—with each other, with our users, and with the world at large. Going above and beyond for each other, our customers, and all of the people affected by our technology is meeting expectations.
Ignite a race to the top on safety
As a safety-first company, we believe that building reliable, trustworthy, and secure systems is our collective r

 ### Link: product overview page
The AI for Problem Solvers | Claude by Anthropic

Skip to main content
Product
Products
Claude
Claude Code
@Claude
Specialized
Claude Security
Claude Science
Capabilities
Artifacts
Design
Connectors
Plugins
Skills
Extensions
Claude for Chrome
Claude for Microsoft 365
Models
Mythos
Fable
Opus
Sonnet
Haiku
Import to Claude
Download apps
Login
(opens in new tab)
Developers
Build with
Claude Code
Claude Platform
For developers
Claude Academy
Community
Developer docs
Console
Enterprise
Enterprise
Overview
Claude Code for Enterprise
Claude Platform
Use Cases
AI Agents
Coding
Commerce
Departments
Customer support
Cybersecurity
Legal
Sales
Industries
Financial services
Government
Healthcare
Higher education
K-12 teachers
Life sciences
Nonprofits
Customers
Contact sales
Resources
Insights
Blog
Customer stories
Anthropic news
Learn
Claude Academy
Courses
Tutorials
Use cases
Connect
Events
Community
Pricing
Overview
API
Login
Contact sales
Try Claude
Product
Explore here
Ask questions about this page
Copy as markdown
Latest news
Claude Cowork is now just Claude.
Rolling out to Pro and Max, with more plans to follow.
Read what changed
(opens in new tab)
Campaign readout
Call prep
Audience intelligence
QBR deck
Earnings comps
CRM hygiene
Month-end close
JD check
Budget reforecast
Survey readout
Legal research drafting
Merit analysis
Deposition prep
Action tracking
Contract redline
Metrics dashboard
Asset creation
QBR deck
Campaign readout
Call prep
Audience intelligence
Earnings comps
JD check
Month-end close
Survey readout
Budget reforecast
Merit analysis
Action tracking
Contract redline
Metrics dashboard
Deposition prep
Feedback digest
Give
Claude
more
Hand Claude a task, not just a question. Tackle routines, tangled ideas, and big projects.
Try Claude
Download for macOS
01
Take tasks off your desk
Ask Claude for more than an answer: mine data, pull information together, or make a first draft. Get time back to make a call.
02
Turn ideas into something real
Bring in a half-form

 ### Link: contact sales page
Contact sales | Claude by Anthropic

Product
Products
Claude
Claude Code
@Claude
Specialized
Claude Security
Claude Science
Capabilities
Artifacts
Design
Connectors
Plugins
Skills
Claude apps built for
Claude in Chrome
Claude for Microsoft 365
Models
Mythos
Fable
Opus
Sonnet
Haiku
Import to Claude
Download apps
Download apps
Download apps
Login
Login
Login
Developers
Build with
Claude Code
Claude Platform
For developers
Claude Academy
Community
Developer docs
Developer docs
Developer docs
Console
Console
Console
Enterprise
Enterprise
Overview
Claude Code for Enterprise
Claude Platform
Use cases
AI agents
Coding
Commerce
Departments
Customer support
Cybersecurity
Legal
Sales
Industries
Financial services
Government
Healthcare
Higher education
K-12 teachers
Life sciences
Nonprofits
Customer stories
Customer stories
Customer stories
Contact sales
Contact sales
Contact sales
Resources
Insights
Blog
Customer stories
Anthropic news
Learn
Claude Academy
Courses
Tutorials
Use cases
Connect
Events
Community
Pricing
Overview
API
Login
Contact sales
Contact sales
Contact sales
Try Claude
Try Claude
Try Claude
Contact sales
Contact sales
Contact sales
Try Claude
Try Claude
Try Claude
Contact sales
Contact sales
Contact sales
Try Claude
Try Claude
Try Claude
Contact sales
Contact sales
Contact sales
Try Claude
Try Claude
Try Claude
Product
Products
Claude
Claude Code
@Claude
Specialized
Claude Security
Claude Science
Capabilities
Artifacts
Design
Connectors
Plugins
Skills
Claude apps built for
Claude in Chrome
Claude for Microsoft 365
Models
Mythos
Fable
Opus
Sonnet
Haiku
Import to Claude
Download apps
Download apps
Download apps
Login
Login
Login
Developers
Build with
Claude Code
Claude Platform
For developers
Claude Academy
Community
Developer docs
Developer docs
Developer docs
Console
Console
Console
Enterprise
Enterprise
Overview
Claude Code for Enterprise
Claude Platform
Use cases
AI agents
Coding
Commerce
Departments
Customer support
Cybersecurity
Legal
Sales
Industries
Fi